In [72]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

In [73]:
#data loading
netflix = pd.read_csv('../data/raw/netflix_titles.csv')
imdb = pd.read_csv('../data/raw/Netflix TV Shows and Movies.csv')

print(netflix.shape)
print(netflix.columns)

print(imdb.shape)
print(imdb.columns)

(8807, 12)
Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='str')
(5283, 11)
Index(['index', 'id', 'title', 'type', 'description', 'release_year',
       'age_certification', 'runtime', 'imdb_id', 'imdb_score', 'imdb_votes'],
      dtype='str')


In [74]:
netflix.head(2)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."


In [75]:
imdb.head(2)

,index,id,title,type,description,release_year,age_certification,runtime,imdb_id,imdb_score,imdb_votes
0,0,tm84618,Taxi Driver,MOVIE,A mentally unstable Vietnam War veteran works ...,1976,R,113,tt0075314,8.3,795222.0
1,1,tm127384,Monty Python and the Holy Grail,MOVIE,"King Arthur, accompanied by his squire, recrui...",1975,PG,91,tt0071853,8.2,530877.0


In [76]:
# cleaning titles for merging
netflix['title'] = netflix['title'].str.lower().str.strip()
imdb['title'] = imdb['title'].str.lower().str.strip()

# merge
df = pd.merge(netflix, imdb, on='title', how='inner')

print("Merged shape:", df.shape)
df.head(3)

Merged shape: (3960, 22)


,show_id,type_x,title,director,cast,country,date_added,release_year_x,rating,duration,...,index,id,type_y,description_y,release_year_y,age_certification,runtime,imdb_id,imdb_score,imdb_votes
0,s1,Movie,dick johnson is dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,...,3241,tm845484,MOVIE,"With this inventive portrait, director Kirsten...",2020,PG-13,89,tt11394180,7.4,6390.0
1,s3,TV Show,ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,...,4811,ts304058,SHOW,"Mehdi, a qualified robber, and Liana, an appre...",2021,TV-MA,46,tt13278100,7.0,2460.0
2,s4,TV Show,jailbirds new orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,...,5190,ts308053,SHOW,"Feuds, flirtations and toilet talk go down amo...",2021,TV-MA,41,tt15320436,6.6,205.0


In [77]:
df.columns

Index(['show_id', 'type_x', 'title', 'director', 'cast', 'country',
       'date_added', 'release_year_x', 'rating', 'duration', 'listed_in',
       'description_x', 'index', 'id', 'type_y', 'description_y',
       'release_year_y', 'age_certification', 'runtime', 'imdb_id',
       'imdb_score', 'imdb_votes'],
      dtype='str')

In [78]:
# choose useful columns
df = df[['title', 'type_x', 'country', 'release_year_x', 'duration', 'listed_in', 'rating', 'description_x', 'imdb_score', 'imdb_votes']]

# rename columns
df = df.rename(columns={
    'type_x': 'type',
    'release_year_x': 'release_year',
    'description_x': 'description'
})

print(df.shape)
df.head()

(3960, 10)


,title,type,country,release_year,duration,listed_in,rating,description,imdb_score,imdb_votes
0,dick johnson is dead,Movie,United States,2020,90 min,Documentaries,PG-13,"As her father nears the end of his life, filmm...",7.4,6390.0
1,ganglands,TV Show,NaN,2021,1 Season,"Crime TV Shows, International TV Shows, TV Act...",TV-MA,To protect his family from a powerful drug lor...,7.0,2460.0
2,jailbirds new orleans,TV Show,NaN,2021,1 Season,"Docuseries, Reality TV",TV-MA,"Feuds, flirtations and toilet talk go down amo...",6.6,205.0
3,kota factory,TV Show,India,2021,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",TV-MA,In a city of coaching centers known to train I...,9.3,66985.0
4,midnight mass,TV Show,NaN,2021,1 Season,"TV Dramas, TV Horror, TV Mysteries",TV-MA,The arrival of a charismatic young priest brin...,7.7,102321.0


In [79]:
df['imdb_score'] = pd.to_numeric(df['imdb_score'], errors='coerce')
df['imdb_votes'] = pd.to_numeric(df['imdb_votes'], errors='coerce')
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')

df.isnull().sum()

title             0
type              0
country         298
release_year      0
duration          1
listed_in         0
rating            0
description       0
imdb_score        0
imdb_votes        5
dtype: int64

In [82]:
df = df.dropna(subset=['imdb_score'])
df = df.dropna(subset=['imdb_votes'])
df['country'] = df['country'].fillna('Unknown')

print(df.shape)

(3955, 10)
